# Cypher · Colab

Serveur **OpenAI-compatible** sur une VM Colab gratuite (T4), avec tunnel public,
pour brancher **OpenCode** sur les modèles Cypher.

Adapté de [enescingoz/colab-llm](https://github.com/enescingoz/colab-llm) (Ollama + Cloudflare Tunnel).

**Marche à suivre** : `Exécution → Tout exécuter`, puis copie l'URL affichée à l'étape 6.

> ⚠️ Le tunnel est **public et sans authentification**. Garde la session ouverte
> uniquement le temps de tes tests.


## 1. Choix du modèle

Tous publics sur `huggingface.co/TheShellMaster`.

In [ ]:
# Modèles disponibles : (dépôt, fichier GGUF, quantization, taille)
CYPHER_MODELS = {
    "coder-7b": (
        "TheShellMaster/cypher-Qwen2.5-Coder-7B-Instruct-abliterated",
        "Qwen2.5-Coder-7B-Instruct-abliterated-Q5_K_M.gguf",
        "Q5_K_M", "5,4 Go",
    ),
    "qwen35-9b": (
        "TheShellMaster/cypher-Huihui-Qwen3.5-9B-abliterated",
        "Huihui-Qwen3.5-9B-abliterated.Q5_K_M.gguf",
        "Q5_K_M", "6,5 Go",
    ),
    "vl-8b": (
        "TheShellMaster/cypher-Huihui-Qwen3-VL-8B-Instruct-abliterated",
        "Huihui-Qwen3-VL-8B-Instruct-abliterated.Q5_K_M.gguf",
        "Q5_K_M", "5,9 Go",
    ),
    "qwen35-4b": (
        "TheShellMaster/cypher-Huihui-Qwen3.5-4B-abliterated",
        "Huihui-Qwen3.5-4B-abliterated.Q5_K_M.gguf",
        "Q5_K_M", "3,1 Go",
    ),
    "vl-4b": (
        "TheShellMaster/cypher-Huihui-Qwen3-VL-4B-Instruct-abliterated",
        "Huihui-Qwen3-VL-4B-Instruct-abliterated.Q5_K_M.gguf",
        "Q5_K_M", "2,9 Go",
    ),
    "qwen35-2b": (
        "TheShellMaster/cypher-Huihui-Qwen3.5-2B-abliterated",
        "Huihui-Qwen3.5-2B-abliterated.Q5_K_M.gguf",
        "Q5_K_M", "1,4 Go",
    ),
}

# ─── Change uniquement cette ligne ────────────────────────────────────────────
CHOIX = "coder-7b"
# ─────────────────────────────────────────────────────────────────────────────

REPO_ID, GGUF_FILE, QUANT, TAILLE = CYPHER_MODELS[CHOIX]
MODEL_NAME = f"hf.co/{REPO_ID}:{QUANT}"

%env OLLAMA_CONTEXT_LENGTH=16384
%env OLLAMA_HOST=0.0.0.0
%env OLLAMA_KEEP_ALIVE=-1

print(f"Modèle   : {CHOIX}")
print(f"Dépôt    : {REPO_ID}")
print(f"Fichier  : {GGUF_FILE} ({TAILLE})")
print(f"Référence: {MODEL_NAME}")


## 2. Diagnostic de la VM

In [ ]:
!apt-get install -y -qq lshw pciutils > /dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"RAM disponible : {ram_gb:.1f} Go")
if ram_gb < 20:
    print("⚠️ Pas de runtime High-RAM : le modèle peut manquer de mémoire.")


## 3. Installation d'Ollama

`zstd` est requis par l'installateur pour décompresser l'archive.

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version


## 4. Démarrage d'Ollama et téléchargement du modèle

In [ ]:
import os
import subprocess
import threading
import time

import requests


def start_ollama():
    subprocess.call(["ollama", "serve"])


threading.Thread(target=start_ollama, daemon=True).start()


def wait_for_ollama(timeout=180):
    """Attend que l'API HTTP réponde vraiment (pas seulement le port ouvert)."""
    derniere_erreur = None
    for i in range(timeout):
        try:
            r = requests.get("http://127.0.0.1:11434/api/version", timeout=5)
            if r.status_code == 200:
                print(f"✅ Ollama répond après {i + 1}s (version {r.json().get('version')})")
                return
        except requests.exceptions.RequestException as exc:
            derniere_erreur = exc
        time.sleep(1)
    raise RuntimeError(f"❌ Ollama n'a pas démarré (dernière erreur : {derniere_erreur})")


wait_for_ollama()

# Ollama sait tirer un GGUF directement depuis Hugging Face.
print(f"⏳ téléchargement de {MODEL_NAME} (plusieurs minutes)…", flush=True)
pull = subprocess.run(["ollama", "pull", MODEL_NAME], timeout=3600)
if pull.returncode != 0:
    print("⚠️ Pull Hugging Face impossible, repli sur un téléchargement direct.")
    subprocess.run(["pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import hf_hub_download

    gguf = hf_hub_download(
        repo_id=REPO_ID,
        filename=GGUF_FILE,
        token=os.environ.get("HF_TOKEN") or None,
    )
    with open("/content/Modelfile", "w") as handle:
        handle.write(f"FROM {gguf}\nPARAMETER num_ctx 16384\nPARAMETER temperature 0.2\n")
    local_name = f"cypher-{CHOIX}"
    subprocess.run(["ollama", "create", local_name, "-f", "/content/Modelfile"], check=True)
    MODEL_NAME = local_name
    print(f"✅ Modèle local créé : {MODEL_NAME}")

print("✅ Modèle prêt :", MODEL_NAME)
!ollama list


## 5. Installation de cloudflared

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


## 6. Tunnel public

In [ ]:
import re

cloudflared_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
    if match:
        public_url = match.group(1)
        break

if not public_url:
    raise RuntimeError("❌ Aucune URL Cloudflare obtenue")

print(f"\n✅ URL publique : {public_url}")


## 7. Test de l'endpoint

In [ ]:
payload = {
    "model": MODEL_NAME,
    "prompt": "Réponds uniquement : OK",
    "stream": False,
}
reponse = requests.post(f"{public_url}/api/generate", json=payload, timeout=300)
print(reponse.status_code, reponse.json().get("response", reponse.text)[:300])


## 8. Configuration pour OpenCode

Copie ce JSON dans le `opencode.json` de ton projet.

In [ ]:
import json

config = {
    "$schema": "https://opencode.ai/config.json",
    "model": f"cypher/{CHOIX}",
    "permission": "allow",
    "providers": {
        "cypher": {
            "name": "Cypher Colab",
            "package": "@opencode/ai/providers/openai-compatible",
            "settings": {
                "baseURL": f"{public_url}/v1",
                "apiKey": "ollama",
                "headerTimeout": 300000,
                "chunkTimeout": 300000,
            },
            "models": {
                CHOIX: {
                    "modelID": MODEL_NAME,
                    "name": f"Cypher {CHOIX} (Colab T4)",
                    "capabilities": {"tools": True, "input": ["text"], "output": ["text"]},
                    "limit": {"context": 16384, "output": 2048},
                    "compatibility": {"maxTokensField": "max_tokens", "streamUsage": False},
                }
            },
        }
    },
}
print(json.dumps(config, indent=2, ensure_ascii=False))


## 9. Garder la session vivante

Ne ferme pas cet onglet : le tunnel meurt avec la VM. Pour arrêter proprement,
interromps l'exécution (`Exécution → Interrompre`).
